In [1]:
"""
1000x1000 matrix multiplication, 5 ways.

  1. Pure Python triple loop            (interpreted, element by element)
  2. NumPy, row-by-row                  (1000 calls of A[i] @ B)
  3. NumPy vectorised  A @ B            (one call, dispatches to BLAS dgemm)
  4. BLAS dgemm called directly         (scipy.linalg.blas.dgemm, no NumPy dispatch layer)
  5. JAX jit (XLA-compiled)             (compile once, then reuse)

Timing: time.perf_counter, 10-run average for methods 2-5.
Pure Python takes minutes, so it is timed with fewer runs (see PURE_RUNS).
"""

import os, time, random, platform
import numpy as np
from scipy.linalg import blas
from threadpoolctl import threadpool_info, threadpool_limits

N = 1000
PURE_RUNS = 1          # pure Python at n=1000 is ~1e9 inner iterations; 1 run is enough
FAST_RUNS = 10


def env_info():
    print("Python :", platform.python_version())
    print("NumPy  :", np.__version__)
    print("CPU cores visible:", os.cpu_count())
    for lib in threadpool_info():
        print(f"BLAS   : {lib['internal_api']} {lib['version']} | threads={lib['num_threads']} | arch={lib.get('architecture')}")


# ---------- implementations ----------
def matmul_pure(A, B, n):
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        Ai, Ci = A[i], C[i]
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += Ai[k] * B[k][j]
            Ci[j] = s
    return C


def matmul_np_rows(A, B):
    n = A.shape[0]
    C = np.empty((n, B.shape[1]))
    for i in range(n):
        C[i] = A[i] @ B
    return C


def matmul_np_vec(A, B):
    return A @ B


def matmul_blas_direct(A, B):
    # dgemm expects Fortran-ordered inputs for zero-copy; use the transpose identity
    # C = A @ B  <=>  C^T = B^T @ A^T, and a C-ordered matrix is a F-ordered transpose.
    return blas.dgemm(1.0, B.T, A.T).T


def timed(fn, runs):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return np.array(times) * 1000  # ms


def main():
    env_info()
    rng = np.random.default_rng(0)
    A = rng.random((N, N))
    B = rng.random((N, N))

    # ---- correctness on a small size first (all methods must agree) ----
    n_small = 60
    a, b = A[:n_small, :n_small].copy(), B[:n_small, :n_small].copy()
    ref = a @ b
    assert np.allclose(np.array(matmul_pure(a.tolist(), b.tolist(), n_small)), ref)
    assert np.allclose(matmul_np_rows(a, b), ref)
    assert np.allclose(matmul_blas_direct(a, b), ref)
    print("\nCorrectness: all implementations agree with A @ B (n=60)")

    results = {}

    # ---- 1. pure Python ----
    Al, Bl = A.tolist(), B.tolist()
    t = timed(lambda: matmul_pure(Al, Bl, N), PURE_RUNS)
    results["Pure Python nested loops"] = t

    # ---- 2-4. NumPy variants ----
    results["NumPy row-by-row (A[i] @ B)"] = timed(lambda: matmul_np_rows(A, B), FAST_RUNS)
    matmul_np_vec(A, B)  # warm caches
    results["NumPy vectorised (A @ B)"] = timed(lambda: matmul_np_vec(A, B), FAST_RUNS)
    matmul_blas_direct(A, B)
    results["BLAS dgemm called directly"] = timed(lambda: matmul_blas_direct(A, B), FAST_RUNS)

    # ---- 5. JAX jit ----
    try:
        import jax, jax.numpy as jnp
        jax.config.update("jax_enable_x64", True)
        f = jax.jit(lambda x, y: x @ y)
        Aj, Bj = jnp.asarray(A), jnp.asarray(B)
        t0 = time.perf_counter(); f(Aj, Bj).block_until_ready()
        compile_ms = (time.perf_counter() - t0) * 1000
        results["JAX jit (after warmup)"] = timed(lambda: f(Aj, Bj).block_until_ready(), FAST_RUNS)
        print(f"JAX first call (trace + compile + run): {compile_ms:.1f} ms")
    except ImportError:
        print("JAX not installed - skipping")

    # ---- report ----
    print(f"\n{'Method':<32}{'runs':>5}{'mean (ms)':>14}{'std (ms)':>12}")
    print("-" * 63)
    for name, t in results.items():
        print(f"{name:<32}{len(t):>5}{t.mean():>14.2f}{t.std():>12.2f}")

    slowest = max(v.mean() for v in results.values())
    fastest = min(v.mean() for v in results.values())
    print(f"\nFastest vs slowest: {slowest / fastest:,.0f}x")
    pure = results["Pure Python nested loops"].mean()
    for name, t in results.items():
        print(f"  {name:<32} {pure / t.mean():>12,.1f}x faster than pure Python")

    # ---- BLAS thread scaling (only meaningful with >1 core) ----
    print("\nBLAS thread scaling for A @ B:")
    for th in sorted({1, 2, 4, os.cpu_count()}):
        if th > os.cpu_count():
            continue
        with threadpool_limits(limits=th, user_api="blas"):
            t = timed(lambda: matmul_np_vec(A, B), FAST_RUNS)
        print(f"  threads={th}: {t.mean():.2f} ms")

    # GFLOPS sanity check: 2*n^3 floating point ops
    flops = 2 * N ** 3
    best = results["NumPy vectorised (A @ B)"].mean() / 1000
    print(f"\nA @ B sustained throughput: {flops / best / 1e9:.1f} GFLOPS")


if __name__ == "__main__":
    main()


Python : 3.12.3
NumPy  : 2.4.4
CPU cores visible: 1
BLAS   : openblas 0.3.31.188.0 | threads=1 | arch=SkylakeX
BLAS   : openblas 0.3.30 | threads=1 | arch=SkylakeX

Correctness: all implementations agree with A @ B (n=60)


JAX first call (trace + compile + run): 44.6 ms

Method                           runs     mean (ms)    std (ms)
---------------------------------------------------------------
Pure Python nested loops            1      76491.04        0.00
NumPy row-by-row (A[i] @ B)        10        262.80        9.87
NumPy vectorised (A @ B)           10         27.72        1.77
BLAS dgemm called directly         10         28.84        1.03
JAX jit (after warmup)             10         27.62        1.58

Fastest vs slowest: 2,769x
  Pure Python nested loops                  1.0x faster than pure Python
  NumPy row-by-row (A[i] @ B)             291.1x faster than pure Python
  NumPy vectorised (A @ B)              2,759.5x faster than pure Python
  BLAS dgemm called directly            2,651.8x faster than pure Python
  JAX jit (after warmup)                2,769.3x faster than pure Python

BLAS thread scaling for A @ B:


  threads=1: 26.04 ms

A @ B sustained throughput: 72.2 GFLOPS
